# Deep Hedging — Phase 1, brique 2 : prix et delta de Black-Scholes

On code le prix et le delta du call européen (les formules de Hull), et surtout on **prouve** qu'ils sont corrects par trois vérifications avant de les brancher sur les trajectoires.

Convention : on paramètre par `tau`, le **temps restant jusqu'à maturité**, pas la date absolue. C'est ce dont on aura besoin pour la couverture, où `tau = T - t` décroît le long de la trajectoire.

Rappel des formules :

    d1 = [ ln(S/K) + (r + sigma²/2) tau ] / (sigma sqrt(tau))
    d2 = d1 - sigma sqrt(tau)
    Call  = S N(d1) - K e^{-r tau} N(d2)
    delta_call = N(d1)

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

In [ ]:
def bs_price(S, K, tau, r, sigma, kind="call"):
    """Prix Black-Scholes d'un call ou put européen. tau = temps restant."""
    S = np.asarray(S, dtype=float)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    d2 = d1 - sigma * np.sqrt(tau)
    if kind == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)
    else:
        return K * np.exp(-r * tau) * norm.cdf(-d2) - S * norm.cdf(-d1)


def bs_delta(S, K, tau, r, sigma, kind="call"):
    """Delta Black-Scholes = dPrix/dS. Pour un call, c'est N(d1)."""
    S = np.asarray(S, dtype=float)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    return norm.cdf(d1) if kind == "call" else norm.cdf(d1) - 1.0

## Vérification 1 : put-call parity

Sans aucun modèle, l'absence d'arbitrage impose `C - P = S - K e^{-r tau}`. Si nos deux fonctions la respectent, elles sont mutuellement cohérentes.

In [ ]:
K, tau, r, sigma = 100.0, 1.0, 0.02, 0.20
for S in [80, 100, 120]:
    C = bs_price(S, K, tau, r, sigma, "call")
    P = bs_price(S, K, tau, r, sigma, "put")
    print(f"S={S}: C-P = {C-P:.6f}   S - K e^-r tau = {S - K*np.exp(-r*tau):.6f}")

## Vérification 2 : delta est bien la dérivée du prix

On compare le delta analytique `N(d1)` à la dérivée numérique `(C(S+h) - C(S-h)) / (2h)`. S'ils coïncident, c'est la preuve sur machine que `delta = dC/dS`, l'objet qui annule le terme en dS dans la couverture.

In [ ]:
h = 1e-4
for S in [80, 100, 120]:
    fd = (bs_price(S+h, K, tau, r, sigma) - bs_price(S-h, K, tau, r, sigma)) / (2*h)
    an = bs_delta(S, K, tau, r, sigma)
    print(f"S={S}: delta analytique = {an:.6f}   diff finie = {fd:.6f}   écart = {abs(an-fd):.2e}")

## Vérification 3 : limites du delta

Très dans la monnaie, le call se comporte comme l'actif, delta proche de 1. Très hors de la monnaie, delta proche de 0. À la monnaie, autour de 0.5.

In [ ]:
for S, lab in [(200,"très ITM"), (100,"ATM"), (40,"très OTM")]:
    print(f"{lab:9s} (S={S}): delta = {bs_delta(S,K,tau,r,sigma):.4f}")

## Visualisation : prix et delta en fonction de S

À gauche, le prix du call pour deux maturités, avec le payoff (tau=0) en pointillés. À droite, le delta. Remarque comme le delta devient plus **raide** quand tau diminue : c'est le gamma qui explose près de la monnaie à l'approche de l'échéance, et c'est lui qui rendra la couverture discrète difficile.

In [ ]:
Sgrid = np.linspace(50, 150, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for tt, c in [(1.0, "navy"), (0.1, "crimson")]:
    ax1.plot(Sgrid, bs_price(Sgrid, K, tt, r, sigma), c, label=f"tau={tt}")
    ax2.plot(Sgrid, bs_delta(Sgrid, K, tt, r, sigma), c, label=f"tau={tt}")
ax1.plot(Sgrid, np.maximum(Sgrid-K, 0), "k--", lw=1, label="payoff (tau=0)")
ax1.set_title("Prix du call vs S"); ax1.set_xlabel("S"); ax1.legend()
ax2.set_title("Delta du call vs S"); ax2.set_xlabel("S"); ax2.set_ylabel("delta"); ax2.legend()
plt.tight_layout(); plt.show()